In [0]:
USE SCHEMA electiva;

Se revisa que esten los 8 archivos de datos 


In [0]:
LIST '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/';

In [0]:
CREATE OR REPLACE TABLE electiva.customers
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/customers.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
CREATE OR REPLACE TABLE electiva.employee
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/employees.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
CREATE OR REPLACE TABLE electiva.orders
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/orders.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
CREATE OR REPLACE TABLE electiva.product_categories
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/productcategories.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
CREATE OR REPLACE TABLE electiva.product_sub_categories
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/productsubcategories.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
CREATE OR REPLACE TABLE electiva.products
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/products.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
CREATE OR REPLACE TABLE electiva.vendor
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/vendors.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

In [0]:
CREATE OR REPLACE TABLE electiva.vendor_product
USING DELTA
AS
SELECT *
FROM read_files(
  '/Volumes/workspace/electiva/mi_volumen_datos/datos~1/vendorproduct.csv',
  format => 'csv',
  header => true,
  inferSchema => true
);

**Punto 1. Construcción del nombre de usuario**

In [0]:
SELECT
    CustomerID,
    FullName,
    CONCAT(
        CAST(CustomerID AS STRING), '-',
        UPPER(SUBSTRING(FirstName, 1, 2)), '-',
        LOWER(SUBSTRING(LastName, 1, 2)), '-AW'
    ) AS user_name
FROM electiva.customers;

**Punto 2. Creación de la tabla page_user**

In [0]:
CREATE TABLE electiva.page_user (
    customerid INT,
    user_name STRING
);

Insertamos los datos de los clientes en page_user

In [0]:
INSERT INTO electiva.page_user

SELECT
    CustomerID,
    CONCAT(
        CAST(CustomerID AS STRING), '-',
        UPPER(SUBSTRING(FirstName, 1, 2)), '-',
        LOWER(SUBSTRING(LastName, 1, 2)), '-AW'
    ) AS user_name
FROM electiva.customers;

In [0]:
SELECT * FROM electiva.page_user;

**Punto 3. Órdenes con descuentos**

In [0]:
-- Consultamos la estructura de la tabla orders

DESCRIBE electiva.orders;

In [0]:
-- Consultamos la estructura de la tabla products

DESCRIBE electiva.products;

In [0]:
-- Consultamos la estructura de las subcategorías

DESCRIBE electiva.product_sub_categories;

In [0]:
-- Consultamos la estructura de las categorías
DESCRIBE electiva.product_categories;

In [0]:
-- Creamos la vista con las órdenes y sus descuentos
CREATE OR REPLACE VIEW electiva.`descuento_mitad_año` AS

WITH ordenes AS (
    SELECT *
    FROM electiva.orders
    WHERE OrderDate >= '2011-08-01'
      AND OrderDate < '2011-09-01'
)

SELECT
    p.ProductName AS productname,
    p.ModelName AS modelname,
    u.user_name,
    o.SalesOrderID AS salesorderid,
    c.CategoryName AS categoryname,
    o.OrderDate AS orderdate,
    o.LineTotal AS linetotal,

    CASE
        WHEN c.CategoryName ILIKE '%Bikes%' THEN 0.10
        WHEN c.CategoryName ILIKE '%Clothing%' THEN 0.15
        WHEN c.CategoryName ILIKE '%Components%' THEN 0.08
        WHEN c.CategoryName ILIKE '%Accessories%' THEN 0.20
        ELSE 0
    END AS descuento,

    o.LineTotal *
    CASE
        WHEN c.CategoryName ILIKE '%Bikes%' THEN 0.10
        WHEN c.CategoryName ILIKE '%Clothing%' THEN 0.15
        WHEN c.CategoryName ILIKE '%Components%' THEN 0.08
        WHEN c.CategoryName ILIKE '%Accessories%' THEN 0.20
        ELSE 0
    END AS total_descuentos_aplicados

FROM ordenes o
INNER JOIN electiva.products p
    ON o.ProductID = p.ProductID
INNER JOIN electiva.product_sub_categories sc
    ON p.SubCategoryID = sc.SubCategoryID
INNER JOIN electiva.product_categories c
    ON sc.CategoryID = c.CategoryID
INNER JOIN electiva.page_user u
    ON o.CustomerID = u.customerid;

In [0]:
SELECT *
FROM electiva.`descuento_mitad_año`;

**Punto 4. Clasificación de clientes**
En este punto se clasifican los clientes según la cantidad de órdenes realizadas durante 2011, asignándoles una categoría de acuerdo con su nivel de compras.

In [0]:
CREATE OR REPLACE MATERIALIZED VIEW electiva.customer_class AS

WITH ordenes_cliente AS (
    SELECT
        CustomerID,
        COUNT(DISTINCT SalesOrderID) AS amount_of_orders
    FROM electiva.orders
    WHERE OrderDate >= '2011-01-01'
      AND OrderDate < '2012-01-01'
    GROUP BY CustomerID
)

SELECT
    COALESCE(o.amount_of_orders, 0) AS amount_of_orders,
    p.user_name,
    CONCAT(c.FirstName, ' ', c.LastName) AS customerfullname,

    CASE
        WHEN COALESCE(o.amount_of_orders, 0) >= 3 THEN 'Oro'
        WHEN COALESCE(o.amount_of_orders, 0) = 2 THEN 'Plata'
        ELSE 'Ocasional'
    END AS user_level,

    c.CustomerID AS customerid

FROM electiva.customers c
INNER JOIN electiva.page_user p
    ON c.CustomerID = p.customerid
LEFT JOIN ordenes_cliente o
    ON c.CustomerID = o.CustomerID;

In [0]:
SELECT *
FROM electiva.customer_class;

Yulian Jainover Montilla Velasco 